# TP1 — Statistiques sur des données météo

**Base de données :** `meteo_dakar_2025.csv`  
**Ville :** Dakar  
**Année :** 2025

## Objectifs
1. Créer l'environnement et le notebook
2. Charger les températures de Dakar
3. Calculer les moyennes, extrêmes et jours de canicule par mois
4. Normaliser les données
5. Comparer une boucle Python et NumPy avec `%timeit`


## 1. Créer l'environnement et le notebook

Ce notebook est conçu pour **Google Colab**. Les bibliothèques nécessaires sont installées/importées dans les cellules suivantes.

> Pour utiliser la base de données, téléversez `meteo_dakar_2025.csv` dans Colab avant d'exécuter la cellule de chargement.


In [ ]:
# Installation / vérification des bibliothèques
!pip -q install numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy :", np.__version__)
print("Pandas :", pd.__version__)


### Importer la base de données

Dans Google Colab, la cellule suivante ouvre une fenêtre permettant de sélectionner le fichier `meteo_dakar_2025.csv`.


In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
# Chargement du fichier CSV
import io

filename = "meteo_dakar_2025.csv"

data = pd.read_csv(filename)

print("Dimensions :", data.shape)
display(data.head())


## 2. Générer ou charger les températures de Dakar

Nous avons choisi de **charger les données réelles fournies dans le fichier CSV** plutôt que de générer artificiellement les températures.

Nous vérifions ensuite :
- les colonnes ;
- les types de données ;
- les valeurs manquantes ;
- les statistiques générales.


In [ ]:
# Informations générales
print("Colonnes :", list(data.columns))
print("\nInformations :")
data.info()

print("\nValeurs manquantes :")
display(data.isnull().sum())

print("\nStatistiques numériques :")
display(data.describe())


In [ ]:
# Préparation des colonnes date et température
date_col = 'date'
temp_col = 'temp_max_c'

print("Colonne date :", date_col)
print("Colonne température :", temp_col)

data[date_col] = pd.to_datetime(data[date_col])
data[temp_col] = pd.to_numeric(data[temp_col], errors="coerce")

# Suppression des lignes sans date ou température
data = data.dropna(subset=[date_col, temp_col]).copy()

# Création du mois
data["mois_num"] = data[date_col].dt.month
data["nom_mois"] = data[date_col].dt.month_name()

display(data.head())


In [ ]:
# Visualisation des températures quotidiennes
plt.figure(figsize=(12, 5))
plt.plot(data[date_col], data[temp_col])
plt.title("Températures quotidiennes à Dakar en 2025")
plt.xlabel("Date")
plt.ylabel("Température (°C)")
plt.grid()
plt.show()


## 3. Moyennes, extrêmes et jours de canicule par mois

Pour ce TP, on définit un **jour de canicule** comme une journée où la température est **supérieure ou égale à 35 °C**.

Nous allons calculer pour chaque mois :
- la température moyenne ;
- la température minimale ;
- la température maximale ;
- le nombre de jours de canicule.


In [ ]:
# Définition d'un jour de canicule
SEUIL_CANICULE = 35

data["canicule"] = data[temp_col] >= SEUIL_CANICULE

# Statistiques mensuelles
statistiques = data.groupby("mois_num")[temp_col].agg(
    moyenne="mean",
    minimum="min",
    maximum="max"
)

# Nombre de jours de canicule
jours_canicule = data.groupby("mois_num")["canicule"].sum()

statistiques["jours_canicule"] = jours_canicule

noms_mois = {
    1: "Janvier", 2: "Février", 3: "Mars", 4: "Avril",
    5: "Mai", 6: "Juin", 7: "Juillet", 8: "Août",
    9: "Septembre", 10: "Octobre", 11: "Novembre", 12: "Décembre"
}

statistiques.index = statistiques.index.map(noms_mois)

display(statistiques.round(2))


In [ ]:
# Mois ayant la température moyenne la plus élevée
mois_plus_chaud = statistiques["moyenne"].idxmax()

print("Mois le plus chaud :", mois_plus_chaud)
print("Température moyenne :", round(statistiques.loc[mois_plus_chaud, "moyenne"], 2), "°C")

# Température maximale de toute l'année
idx_max = data[temp_col].idxmax()

print("\nTempérature maximale annuelle :", data.loc[idx_max, temp_col], "°C")
print("Date :", data.loc[idx_max, date_col].date())


In [ ]:
# Graphique : température moyenne par mois
plt.figure(figsize=(11, 5))
plt.bar(statistiques.index, statistiques["moyenne"])
plt.title("Température moyenne par mois à Dakar")
plt.xlabel("Mois")
plt.ylabel("Température moyenne (°C)")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()


In [ ]:
# Graphique : nombre de jours de canicule
plt.figure(figsize=(11, 5))
plt.bar(statistiques.index, statistiques["jours_canicule"])
plt.title("Nombre de jours de canicule par mois")
plt.xlabel("Mois")
plt.ylabel("Nombre de jours (≥ 35 °C)")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()


## 4. Normaliser les données

Nous utilisons ici la **normalisation Min-Max** :

\[
x_{normalisé} = \frac{x - x_{min}}{x_{max} - x_{min}}
\]

Après normalisation, les températures sont généralement comprises entre **0 et 1**.

Cette opération est particulièrement utile en apprentissage automatique lorsque les variables ont des échelles différentes.


In [ ]:
# Normalisation Min-Max avec NumPy
temperatures = data[temp_col].to_numpy(dtype=float)

temp_min = temperatures.min()
temp_max = temperatures.max()

data["temperature_normalisee"] = (
    (temperatures - temp_min) / (temp_max - temp_min)
)

display(data[[date_col, temp_col, "temperature_normalisee"]].head())

print("Minimum normalisé :", data["temperature_normalisee"].min())
print("Maximum normalisé :", data["temperature_normalisee"].max())


## 5. Comparer une boucle Python et NumPy avec `%timeit`

Nous allons effectuer la même normalisation de deux façons :

1. avec une **boucle Python** ;
2. avec une opération **vectorisée NumPy**.

`%timeit` exécute plusieurs fois une instruction afin d'obtenir une mesure plus fiable du temps d'exécution.


In [ ]:
# Préparation d'un tableau plus grand pour rendre la comparaison visible
temperatures_test = np.tile(temperatures, 1000)

temp_min_test = temperatures_test.min()
temp_max_test = temperatures_test.max()

temperatures_liste = temperatures_test.tolist()

def normalisation_python(liste):
    minimum = min(liste)
    maximum = max(liste)
    return [(x - minimum) / (maximum - minimum) for x in liste]

def normalisation_numpy(tableau):
    minimum = tableau.min()
    maximum = tableau.max()
    return (tableau - minimum) / (maximum - minimum)


In [ ]:
# Comparaison des temps d'exécution
print("Boucle Python :")
%timeit normalisation_python(temperatures_liste)

print("\nNumPy :")
%timeit normalisation_numpy(temperatures_test)


In [ ]:
# Vérification que les deux méthodes donnent le même résultat
resultat_python = np.array(normalisation_python(temperatures_liste))
resultat_numpy = normalisation_numpy(temperatures_test)

print("Résultats identiques :", np.allclose(resultat_python, resultat_numpy))


# Conclusion

Dans ce TP, nous avons chargé les températures de Dakar pour 2025 et réalisé plusieurs traitements statistiques.

Nous avons calculé les températures moyennes, minimales et maximales par mois ainsi que le nombre de jours dépassant le seuil de canicule de 35 °C. Nous avons ensuite appliqué une normalisation Min-Max afin de ramener les températures sur une échelle comparable.

Enfin, la comparaison avec `%timeit` permet de constater l'intérêt de la **vectorisation NumPy** pour effectuer rapidement des calculs sur de nombreux éléments, par rapport à une boucle Python classique.
